In [1]:
import os
os.chdir('/workspace/d83bce0e-6a61-424a-860f-cf53ff19fe12')
print(os.listdir('.'))


['-PROMPT-v6-DATASET.md', '.prompts', 'memory', '.config', 'delta_worker.py', '.kernel_llm_logs_1.txt']


In [2]:
import subprocess
# Test that cypari2 works and how lfunmf+lfunzeros performs on small interval
test_code = '''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 20])
print("elapsed:", time.time()-t0)
print("n:", len(z))
for i in range(min(3, len(z))):
 # try Strprintf
 s = str(pari(f'Strprintf("%.80g", lfunzeros(L,[0,20])[{i+1}])'))
 print(i, s)
'''
r = subprocess.run(['python', '-c', test_code], capture_output=True, text=True, timeout=300)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-2000:] if r.stderr else "")


STDOUT: 
STDERR: Traceback (most recent call last):
 File "<string>", line 2, in <module>
 import cypari2, time
ModuleNotFoundError: No module named 'cypari2'



In [3]:
import subprocess
r = subprocess.run(['pip', 'install', 'cypari2', 'cysignals'], capture_output=True, text=True, timeout=600)
print(r.stdout[-2000:])
print("STDERR:", r.stderr[-2000:])



STDERR: 


In [4]:
import subprocess
test_code = '''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 20])
print("elapsed:", time.time()-t0)
print("n:", len(z))
# Assign z to PARI variable so we can format from PARI side
pari("Z = lfunzeros(L,[0,20])")
for i in range(min(3, len(z))):
 s_default = str(z[i])
 s_hp = str(pari(f"Strprintf(\\"%.80g\\", Z[{i+1}])"))
 print(i, "default:", s_default)
 print(i, "hp:", s_hp)
'''
r = subprocess.run(['python', '-c', test_code], capture_output=True, text=True, timeout=600)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-2000:])


STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
elapsed: 0.005652904510498047
n: 4
0 default: 9.222379399921102523
0 hp: 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290
1 default: 13.907549861392134407
1 hp: 13.907549861392134406446681328770219491757552353514488133240529496565487494434514
2 default: 17.442776978234473314
2 hp: 17.442776978234473313551525137127262718708866524275272068938570762442661986293838

STDERR: 


In [5]:
# Quick benchmark: how long does it take to find zeros up to T=4075? Test on a small chunk
import subprocess
test_code = '''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
# Test on a chunk near the higher end
z = pari.lfunzeros(pari("L"), [4000, 4100])
print("elapsed [4000,4100]:", time.time()-t0, "n:", len(z))
'''
r = subprocess.run(['python', '-c', test_code], capture_output=True, text=True, timeout=900)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-2000:])


TimeoutError: Code execution timed out after 900.0 seconds

In [6]:
# That timed out. Let's check zero density and aim for the cube root spacing.
# The cube-root strategy means each shard's interval is [T*(i/8)^{1/3}, T*((i+1)/8)^{1/3}]
# This makes higher intervals smaller, balancing the work since zero density grows.
import subprocess, time
# Cancel kernel-side; restart fresh


In [7]:
# Let's just check killing leftover python processes first
import subprocess
r = subprocess.run(['pgrep', '-fa', 'python'], capture_output=True, text=True)
print(r.stdout)


PermissionError: [Errno 13] Permission denied: 'pgrep'

In [8]:
# Compute partition with cube-root spacing
T_max = 4075
import numpy as np
edges = [T_max * (i/8)**(1/3) for i in range(9)]
print(edges)
widths = [edges[i+1]-edges[i] for i in range(8)]
print("widths:", widths)


[0.0, 2037.5, 2567.0891391608043, 2938.5834995013442, 3234.3296433852065, 3484.0759913537704, 3702.3832078954847, 3897.597284898743, 4075.0]
widths: [2037.5, 529.5891391608043, 371.49436034053997, 295.74614388386226, 249.7463479685639, 218.3072165417143, 195.21407700325835, 177.40271510125694]


In [9]:
# That's the cube-root spacing. The first shard [0, 2037.5] is the largest but lowest-density.
# Let me first verify that lfunzeros can handle large intervals - test [0, 100] quickly
import subprocess, time
test_code = '''
import cypari2, time, sys
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 100])
print("[0,100]:", time.time()-t0, "n:", len(z), flush=True)
'''
r = subprocess.run(['python', '-c', test_code], capture_output=True, text=True, timeout=600)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-1500:])


STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[0,100]: 0.09876728057861328 n: 59

STDERR: 


In [10]:
# 59 zeros up to T=100 in 0.1s. Modular form L(Delta,s) has gamma factor Gamma_C(s+11/2) so 
# zero density ~ (1/2pi) log(T/(2pi)) * 2 ... For weight 12 the analytic conductor scales differently.
# At T=100, density n/T = 0.59. We need 5000 zeros, so T_max ~ 4075 sounds about right.
# But [4000,4100] timed out at 900s - that's strange. Let me try a smaller chunk near T=4000.
import subprocess, time
test_code = '''
import cypari2, time, sys
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
z = pari.lfunzeros(pari("L"), [3900, 3920])
print("[3900,3920]:", time.time()-t0, "n:", len(z), flush=True)
'''
r = subprocess.run(['python', '-c', test_code], capture_output=True, text=True, timeout=600)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-1500:])


STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[3900,3920]: 283.6995053291321 n: 40

STDERR: 


In [11]:
# So at T~3900, 20 units of T takes ~284 seconds. The width of shard 7 is 177 units, which would 
# take roughly ~2500 seconds = 42 minutes per shard if linear. Running 8 in parallel, the bottleneck 
# is the largest shard. The earliest shard [0,2037.5] could also be slow due to its width even though 
# zeros are sparser - need to estimate.
# Estimate timing roughly: density ~ log(T)/2pi. Cost likely scales as (zeros found) * (something).
# In [3900,3920], 40 zeros in 284s = 7.1s/zero. Total 5000 zeros at maybe 1-7s/zero average?
# At T=100, 59 zeros in 0.1s = 0.0017s/zero. Cost grows.
# Let's estimate cost per zero at a few heights:
import subprocess, time
test_code = '''
import cypari2, time, sys
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
for tlo, thi in [(0,200), (1000,1020), (2000,2020), (3000,3020)]:
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tlo, thi])
 print(f"[{tlo},{thi}]: {time.time()-t0:.2f}s n={len(z)}", flush=True)
'''
r = subprocess.run(['python', '-c', test_code], capture_output=True, text=True, timeout=1500)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-1500:])


STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[0,200]: 0.51s n=159
[1000,1020]: 4.31s n=32
[2000,2020]: 36.70s n=37
[3000,3020]: 124.75s n=40

STDERR: 


In [12]:
# Time per zero appears to grow roughly as T^2 (or similar polynomial). 
# At T=3000, ~3.1s/zero; T=3900, ~7.1s/zero. 
# For shard 7 [3897, 4075], width ~ 178, density ~ 40/20 = 2 zeros/unit -> ~360 zeros, at ~8s/zero -> ~2900s = 48 min
# For shard 0 [0, 2037.5]: zero density at T=2000 is ~1.8/unit -> ~3700 zeros, but at lower T cost is small. 
# Let me estimate by integrating. cost per zero at height T ~ alpha * T^p
# Fit: at T=1000: 4.31/32 = 0.135 s/zero. At T=2000: 36.7/37 = 0.99. At T=3000: 124.75/40 = 3.12. At T=3900: 7.1
# Ratios: 1000->2000: 0.99/0.135 = 7.3 (factor of 2 in T -> factor 7.3 -> exponent ~2.87)
# 2000->3000: 3.12/0.99 = 3.15, ratio T 1.5 -> exponent ~ log(3.15)/log(1.5) = 2.82
# So cost per zero ~ T^2.85
# Total shard 7 work: integral t^2.85 * density(t) dt from 3897 to 4075
# Let's just estimate by approximation. We have 8 shards running in parallel. 
# We have a 3600s wall clock budget. Need to be careful.

# Time budget is 3600 seconds. The biggest shard may take >2000 seconds. 
# Run shards in parallel with cube-root spacing, hope the runtime balances.

# Let me check available CPU count
import os
print("CPU count:", os.cpu_count())


CPU count: 8


In [13]:
# 8 CPUs and 8 shards - perfect. Let me also estimate shard 0 timing.
# Shard 0 = [0, 2037.5]. At T=2000, 36.7s for 20 units * 37 zeros = density of ~1.85/unit.
# Need to integrate density from 0 to 2037.5: total zeros ~ N(T) = (T/2π) log(T/(2π)) * 2 for L(Delta) weight 12... 
# Actually for L(Delta,s): N(T) ≈ (T/π) log(T/(2π e)) approximately.
# At T=2037: N ~ (2037/3.14) * log(2037/(2π*e)) ≈ 648 * log(119.2) ≈ 648 * 4.78 ≈ 3100 zeros
# So shard 0 needs ~3100 zeros, but at lower T, much faster per zero.
# Integral_0^2037 cost ~ integral T^2.85 dT / (typical) - hmm. The integration time depends mostly on the top end.
# Effectively most cost is near the top. So shard 0 cost ~ (2000/3900)^2.85 * shard7_cost ~ 0.14 * 2900 = ~410s? 

# The cube root partitioning is exactly to balance this. By design, total cost per shard should be roughly equal.

# Let me write the worker and main script.

WORKER = r'''#!/usr/bin/env python3
"""delta_shard_runner.py - run lfunzeros on a sub-interval and save results.

Usage: python delta_shard_runner.py <idx> <tmin> <tmax> <dps> <out_pkl>
"""
import sys, time, pickle, os
import cypari2

def main():
 if len(sys.argv) != 6:
 print("usage: delta_shard_runner.py idx tmin tmax dps out_pkl", file=sys.stderr)
 sys.exit(2)
 idx = int(sys.argv[1])
 tmin = float(sys.argv[2])
 tmax = float(sys.argv[3])
 dps = int(sys.argv[4])
 out_pkl = sys.argv[5]

 pari = cypari2.Pari()
 pari.default("realprecision", dps)
 pari.allocatemem(10**9)
 # Define L(Delta,s) via modular form (LMFDB 1.12.a.a)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

 t0 = time.time()
 # Find zeros in [tmin, tmax]; assign to PARI variable for Strprintf access
 pari(f"Z = lfunzeros(L, [{tmin}, {tmax}])")
 n = int(pari("#Z"))
 elapsed = time.time() - t0

 # Format each zero at full precision via Strprintf
 # Use "%.<dps>g" so we keep all significant digits.
 fmt = f"%.{dps}g"
 out = []
 for i in range(1, n + 1):
 s = str(pari(f'Strprintf("{fmt}", Z[{i}])'))
 out.append(s)

 record = {
 "idx": idx,
 "tmin": tmin,
 "tmax": tmax,
 "dps": dps,
 "elapsed": elapsed,
 "n": n,
 "zeros": out,
 }
 with open(out_pkl, "wb") as f:
 pickle.dump(record, f)
 print(f"[shard {idx}] tmin={tmin} tmax={tmax} n={n} elapsed={elapsed:.1f}s -> {out_pkl}", flush=True)

if __name__ == "__main__":
 main()
'''

with open("delta_shard_runner.py", "w") as f:
 f.write(WORKER)
print("wrote delta_shard_runner.py")


wrote delta_shard_runner.py


In [14]:
MAIN = r'''#!/usr/bin/env python3
"""generate_ldelta.py - parallel independent-subprocess generator for L(Delta,s) zeros.

Strategy:
 - Partition [0, T_max] into 8 sub-intervals via cube-root spacing.
 - Launch 8 independent subprocesses of delta_shard_runner.py (each its own Python
 interpreter + cypari2.Pari instance). They communicate ONLY via pickle files,
 sidestepping the multiprocessing.Pool segfault observed when cypari2 is shared
 through a Pool.
 - Wait for all to finish, concatenate, sort, de-duplicate, truncate to N=5000,
 validate gamma_1, save to ldelta_zeros_N5000_dps80.npy as dtype='<U81'.
"""
import os, sys, subprocess, time, pickle, glob
import numpy as np

T_MAX = 4075.0
DPS = 80
N_TARGET = 5000
NUM_SHARDS = 8

SHARD_DIR = "ldelta_shards"
FINAL_NPY = "ldelta_zeros_N5000_dps80.npy"

def partition(T_max, n):
 edges = [T_max * (i / n) ** (1.0/3.0) for i in range(n + 1)]
 return edges

def main():
 os.makedirs(SHARD_DIR, exist_ok=True)
 edges = partition(T_MAX, NUM_SHARDS)
 print("Cube-root partition edges:")
 for i, e in enumerate(edges):
 print(f" edge[{i}] = {e:.6f}")
 print()

 here = os.path.dirname(os.path.abspath(__file__))
 runner = os.path.join(here, "delta_shard_runner.py")

 # Launch all subprocesses
 procs = []
 shard_files = []
 log_files = []
 t_start = time.time()
 for i in range(NUM_SHARDS):
 tmin = edges[i]
 tmax = edges[i+1]
 out_pkl = os.path.join(SHARD_DIR, f"ldelta_shard_{i}.pkl")
 log_path = os.path.join(SHARD_DIR, f"shard_{i}.log")
 shard_files.append(out_pkl)
 log_files.append(log_path)
 lf = open(log_path, "w")
 cmd = [sys.executable, runner, str(i), repr(tmin), repr(tmax), str(DPS), out_pkl]
 print(f"launching shard {i}: [{tmin:.4f}, {tmax:.4f}] -> {out_pkl}")
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
 procs.append((p, lf))

 # Wait for all to complete
 rc = []
 for i, (p, lf) in enumerate(procs):
 ret = p.wait()
 lf.close()
 rc.append(ret)
 print(f"shard {i} returncode={ret} elapsed_total={time.time()-t_start:.1f}s")

 # If any failed, report and exit non-zero
 if any(r != 0 for r in rc):
 print("ERROR: at least one shard failed", file=sys.stderr)
 for i, r in enumerate(rc):
 if r != 0:
 print(f" shard {i} returncode={r}; see {log_files[i]}", file=sys.stderr)
 sys.exit(1)

 # Load and merge
 all_zeros = []
 for i, pkl in enumerate(shard_files):
 with open(pkl, "rb") as f:
 rec = pickle.load(f)
 print(f"shard {i}: tmin={rec['tmin']:.4f} tmax={rec['tmax']:.4f} "
 f"n={rec['n']} elapsed={rec['elapsed']:.1f}s")
 all_zeros.extend(rec["zeros"])

 print(f"\nTotal zeros collected (raw): {len(all_zeros)}")

 # Convert to floats for sorting / dedup; keep strings as canonical representation
 pairs = []
 for s in all_zeros:
 try:
 v = float(s)
 except Exception as e:
 print(f" WARNING: could not parse {s!r}: {e}")
 continue
 pairs.append((v, s))
 pairs.sort(key=lambda x: x[0])

 # De-duplicate (use a tight tolerance since adjacent shards share boundaries)
 dedup = []
 prev_v = None
 TOL = 1e-9
 for v, s in pairs:
 if prev_v is not None and abs(v - prev_v) < TOL:
 continue
 dedup.append((v, s))
 prev_v = v

 print(f"After de-duplication: {len(dedup)}")
 if len(dedup) < N_TARGET:
 print(f"ERROR: only {len(dedup)} unique zeros, need {N_TARGET}", file=sys.stderr)
 sys.exit(2)

 trimmed = [s for (_, s) in dedup[:N_TARGET]]

 # Validate gamma_1
 GAMMA1_REF = "9.22237939992110252224376719274347813552877062243200928999818642800192129537"
 g1 = trimmed[0]
 print(f"\ngamma_1 (computed): {g1}")
 print(f"gamma_1 (reference): {GAMMA1_REF}")
 # Compare first 19 digits
 def digit_str(s, k):
 # strip leading non-digit
 return s[:k+2] # roughly first k digits + decimal
 nmatch = 0
 for a, b in zip(g1, GAMMA1_REF):
 if a == b:
 nmatch += 1
 else:
 break
 print(f"matching leading characters: {nmatch}")

 # Save as <U81 array
 arr = np.array(trimmed, dtype="<U81")
 np.save(FINAL_NPY, arr)
 print(f"\nSaved {arr.shape[0]} zeros to {FINAL_NPY} (dtype={arr.dtype})")
 print(f"First zero: {arr[0]}")
 print(f"Last zero: {arr[-1]}")
 print(f"Total wall time: {time.time()-t_start:.1f}s")

if __name__ == "__main__":
 main()
'''
with open("generate_ldelta.py", "w") as f:
 f.write(MAIN)
print("wrote generate_ldelta.py")


wrote generate_ldelta.py


In [15]:
# Quick sanity check: run runner on a tiny interval just to verify it works correctly
import subprocess, os
os.makedirs("ldelta_shards_test", exist_ok=True)
r = subprocess.run([
 "python", "delta_shard_runner.py", "0", "0.0", "20.0", "80",
 "ldelta_shards_test/test.pkl"
], capture_output=True, text=True, timeout=120)
print("rc:", r.returncode)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr[-1000:])
import pickle
with open("ldelta_shards_test/test.pkl","rb") as f:
 rec = pickle.load(f)
print("record keys:", rec.keys())
print("n:", rec["n"], "elapsed:", rec["elapsed"])
print("first 3 zeros:")
for z in rec["zeros"][:3]:
 print(" ", z, "len=", len(z))


rc: 0
STDOUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[shard 0] tmin=0.0 tmax=20.0 n=4 elapsed=0.0s -> ldelta_shards_test/test.pkl

STDERR: 
record keys: dict_keys(['idx', 'tmin', 'tmax', 'dps', 'elapsed', 'n', 'zeros'])
n: 4 elapsed: 0.01680469512939453
first 3 zeros:
 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290 len= 81
 13.907549861392134406446681328770219491757552353514488133240529496565487494434514 len= 81
 17.442776978234473313551525137127262718708866524275272068938570762442661986293838 len= 81


In [16]:
# Excellent! Strings are exactly 81 chars (matches <U81). gamma_1 matches the reference precisely.
# Now run the main generation script. I need to be mindful of the 3600s budget.
# I'll launch it as a background subprocess so I can monitor.
import subprocess, os, time, glob

# Clean any prior state
import shutil
if os.path.exists("ldelta_shards"):
 shutil.rmtree("ldelta_shards")

# Launch in background, capturing main log
main_log = open("generate_ldelta.log", "w")
proc = subprocess.Popen(
 ["python", "-u", "generate_ldelta.py"],
 stdout=main_log, stderr=subprocess.STDOUT
)
print("Launched generate_ldelta.py PID:", proc.pid)
start = time.time()


Launched generate_ldelta.py PID: 196


In [17]:
# Monitor progress
import time, os, subprocess
def status():
 elapsed = time.time() - start
 print(f"--- elapsed {elapsed:.0f}s, generate process alive: {proc.poll() is None} ---")
 if os.path.exists("generate_ldelta.log"):
 with open("generate_ldelta.log") as f:
 content = f.read()
 print("MAIN LOG (tail):")
 print(content[-2000:])
 # shard logs
 for i in range(8):
 lp = f"ldelta_shards/shard_{i}.log"
 if os.path.exists(lp):
 with open(lp) as f:
 ll = f.read()
 print(f"shard {i} log tail: {ll[-200:]!r}")
status()


--- elapsed 7s, generate process alive: True ---
MAIN LOG (tail):
Cube-root partition edges:
 edge[0] = 0.000000
 edge[1] = 2037.500000
 edge[2] = 2567.089139
 edge[3] = 2938.583500
 edge[4] = 3234.329643
 edge[5] = 3484.075991
 edge[6] = 3702.383208
 edge[7] = 3897.597285
 edge[8] = 4075.000000

launching shard 0: [0.0000, 2037.5000] -> ldelta_shards/ldelta_shard_0.pkl
launching shard 1: [2037.5000, 2567.0891] -> ldelta_shards/ldelta_shard_1.pkl
launching shard 2: [2567.0891, 2938.5835] -> ldelta_shards/ldelta_shard_2.pkl
launching shard 3: [2938.5835, 3234.3296] -> ldelta_shards/ldelta_shard_3.pkl
launching shard 4: [3234.3296, 3484.0760] -> ldelta_shards/ldelta_shard_4.pkl
launching shard 5: [3484.0760, 3702.3832] -> ldelta_shards/ldelta_shard_5.pkl
launching shard 6: [3702.3832, 3897.5973] -> ldelta_shards/ldelta_shard_6.pkl
launching shard 7: [3897.5973, 4075.0000] -> ldelta_shards/ldelta_shard_7.pkl

shard 0 log tail: 'PARI stack size set to 1000000000 bytes, maximum size set to 

In [18]:
import time
time.sleep(300)
status()


--- elapsed 312s, generate process alive: True ---
MAIN LOG (tail):
Cube-root partition edges:
 edge[0] = 0.000000
 edge[1] = 2037.500000
 edge[2] = 2567.089139
 edge[3] = 2938.583500
 edge[4] = 3234.329643
 edge[5] = 3484.075991
 edge[6] = 3702.383208
 edge[7] = 3897.597285
 edge[8] = 4075.000000

launching shard 0: [0.0000, 2037.5000] -> ldelta_shards/ldelta_shard_0.pkl
launching shard 1: [2037.5000, 2567.0891] -> ldelta_shards/ldelta_shard_1.pkl
launching shard 2: [2567.0891, 2938.5835] -> ldelta_shards/ldelta_shard_2.pkl
launching shard 3: [2938.5835, 3234.3296] -> ldelta_shards/ldelta_shard_3.pkl
launching shard 4: [3234.3296, 3484.0760] -> ldelta_shards/ldelta_shard_4.pkl
launching shard 5: [3484.0760, 3702.3832] -> ldelta_shards/ldelta_shard_5.pkl
launching shard 6: [3702.3832, 3897.5973] -> ldelta_shards/ldelta_shard_6.pkl
launching shard 7: [3897.5973, 4075.0000] -> ldelta_shards/ldelta_shard_7.pkl

shard 0 log tail: 'PARI stack size set to 1000000000 bytes, maximum size set t

In [19]:
# Still running. Wait more.
import time
time.sleep(600)
status()


--- elapsed 917s, generate process alive: True ---
MAIN LOG (tail):
Cube-root partition edges:
 edge[0] = 0.000000
 edge[1] = 2037.500000
 edge[2] = 2567.089139
 edge[3] = 2938.583500
 edge[4] = 3234.329643
 edge[5] = 3484.075991
 edge[6] = 3702.383208
 edge[7] = 3897.597285
 edge[8] = 4075.000000

launching shard 0: [0.0000, 2037.5000] -> ldelta_shards/ldelta_shard_0.pkl
launching shard 1: [2037.5000, 2567.0891] -> ldelta_shards/ldelta_shard_1.pkl
launching shard 2: [2567.0891, 2938.5835] -> ldelta_shards/ldelta_shard_2.pkl
launching shard 3: [2938.5835, 3234.3296] -> ldelta_shards/ldelta_shard_3.pkl
launching shard 4: [3234.3296, 3484.0760] -> ldelta_shards/ldelta_shard_4.pkl
launching shard 5: [3484.0760, 3702.3832] -> ldelta_shards/ldelta_shard_5.pkl
launching shard 6: [3702.3832, 3897.5973] -> ldelta_shards/ldelta_shard_6.pkl
launching shard 7: [3897.5973, 4075.0000] -> ldelta_shards/ldelta_shard_7.pkl

shard 0 log tail: 'PARI stack size set to 1000000000 bytes, maximum size set t

In [20]:
# Time pressure - only 1075s left. Let me check the shard pkl files to see progress
import os, glob, time
for i in range(8):
 pkl = f"ldelta_shards/ldelta_shard_{i}.pkl"
 print(f"shard {i}: exists={os.path.exists(pkl)}", end="")
 if os.path.exists(pkl):
 sz = os.path.getsize(pkl)
 print(f" size={sz}")
 else:
 print()
print(f"\nelapsed: {time.time()-start:.0f}s")
print("proc alive:", proc.poll() is None)


shard 0: exists=False
shard 1: exists=False
shard 2: exists=False
shard 3: exists=False
shard 4: exists=False
shard 5: exists=False
shard 6: exists=False
shard 7: exists=False

elapsed: 926s
proc alive: True


In [21]:
# No shards have completed yet. Given the timing test (T~3900 takes ~14s per 1 unit of T),
# the slowest shard (shard 7, width=177) would need ~14s/unit * 177 = ~2500s.
# We're at 926s elapsed and have 1066s left = ~33 minutes more = max ~2000s total.
# It's very unlikely all 8 will finish in time. I need to face the reality that this is too slow.

# Decision: Let it continue running and check at the end. If insufficient, report findings.
# Meanwhile let me check what's going on by looking at /proc to see CPU usage of children.
import subprocess
# can we use ps?
try:
 r = subprocess.run(['ps', '-eo', 'pid,pcpu,etime,cmd'], capture_output=True, text=True, timeout=10)
 lines = [l for l in r.stdout.split('\n') if 'delta_shard' in l or 'python' in l.lower()]
 for l in lines[:20]:
 print(l)
except Exception as e:
 print(e)


[Errno 13] Permission denied: 'ps'


In [22]:
# Wait another large chunk. We have ~1050s left. Wait 800s then assess.
import time
time.sleep(800)
import os
for i in range(8):
 pkl = f"ldelta_shards/ldelta_shard_{i}.pkl"
 print(f"shard {i}: exists={os.path.exists(pkl)}", end="")
 if os.path.exists(pkl):
 sz = os.path.getsize(pkl)
 print(f" size={sz}")
 else:
 print()
print(f"elapsed total: {time.time()-start:.0f}s, proc alive: {proc.poll() is None}")


TimeoutError: Code execution timed out after 448.0 seconds